In [20]:
import glob
import json
import numpy as np


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

# FOLDER = "/rds/general/user/ll1225/home/imperial_irp/extended_evo1/eval_results/final_results/navigation/100_traj/baseline/easy_traj/trajectories"
# FOLDER = "/rds/general/user/ll1225/home/imperial_irp/extended_evo1/eval_results/final_results/navigation/100_traj/baseline/medium_traj/trajectories"
# FOLDER = "/rds/general/user/ll1225/home/imperial_irp/extended_evo1/eval_results/final_results/navigation/100_traj/baseline/hard_traj/trajectories"
# FOLDER = "/rds/general/user/ll1225/home/imperial_irp/extended_evo1/eval_results/final_results/navigation/100_traj/AGVLA/easy_traj/trajectories"
# FOLDER = "/rds/general/user/ll1225/home/imperial_irp/extended_evo1/eval_results/final_results/navigation/100_traj/AGVLA/medium_traj/trajectories"
FOLDER = "/rds/general/user/ll1225/home/imperial_irp/extended_evo1/eval_results/final_results/navigation/100_traj/AGVLA/hard_traj/trajectories"


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def yaw_diff(a, b):
    """Smallest angular difference, in radians."""
    return abs((a - b + np.pi) % (2 * np.pi) - np.pi)


# ------------------------------------------------------------
# Find reference trajectories
# ------------------------------------------------------------

REFERENCE_FILES = glob.glob(
    f"{FOLDER}/*_reference.json"
)

print(f"Reference files found: {len(REFERENCE_FILES)}")


# ------------------------------------------------------------
# Calculate path lengths and rotations
# ------------------------------------------------------------

path_lengths = []
rotations = []


for file in REFERENCE_FILES:

    with open(file) as f:
        data = json.load(f)

    traj = np.array(
        data["reference_trajectory"],
        dtype=float
    )

    # Reference yaw is degrees -> radians
    yaw = np.deg2rad(traj[:, 3])

    # --------------------------------------------------------
    # 3D path length
    # --------------------------------------------------------

    position_diffs = np.diff(
        traj[:, :3],
        axis=0
    )

    distances = np.linalg.norm(
        position_diffs,
        axis=1
    )

    path_length = np.sum(distances)

    path_lengths.append(path_length)

    # --------------------------------------------------------
    # Cumulative rotation
    # --------------------------------------------------------

    rotation = np.sum([
        yaw_diff(a, b)
        for a, b in zip(yaw[:-1], yaw[1:])
    ])

    rotations.append(rotation)


# ------------------------------------------------------------
# Dataset means
# ------------------------------------------------------------

mean_path_length = np.mean(path_lengths)
mean_rotation = np.mean(rotations)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nDataset Reference Statistics")
print("----------------------------")
print(f"Reference trajectories: {len(path_lengths)}")
print(f"Mean path length:       {mean_path_length:.6f} m")
print(f"Mean rotation:          {mean_rotation:.6f} rad")
print(f"Mean rotation:          {np.rad2deg(mean_rotation):.6f} deg")

Reference files found: 100

Dataset Reference Statistics
----------------------------
Reference trajectories: 100
Mean path length:       2.851800 m
Mean rotation:          1.515033 rad
Mean rotation:          86.804988 deg


In [23]:
import os
import glob
import json
import math
import numpy as np


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

# FOLDER = "/rds/general/user/ll1225/home/imperial_irp/extended_evo1/eval_results/final_results/navigation/100_traj/baseline/easy_traj/trajectories"

SUCCESS_DIST = 0.5
SUCCESS_YAW = math.pi / 4

DTH = 1.0
MEAN_PATH_LENGTH = mean_path_length#2.2
MEAN_ROTATION = mean_rotation#1.0


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def yaw_diff(a, b):
    return abs((a - b + np.pi) % (2 * np.pi) - np.pi)


def dtw(seq1, seq2, dist_fn):
    n, m = len(seq1), len(seq2)
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = dist_fn(seq1[i - 1], seq2[j - 1]) + min(
                D[i - 1, j],
                D[i, j - 1],
                D[i - 1, j - 1]
            )

    return D[n, m]


def path_length(traj):
    return np.sum(
        np.linalg.norm(
            np.diff(traj[:, :3], axis=0),
            axis=1
        )
    )


def rotation_length(traj):
    return np.sum(
        np.abs(
            np.array([
                yaw_diff(a, b)
                for a, b in zip(traj[:-1, 3], traj[1:, 3])
            ])
        )
    )


# ------------------------------------------------------------
# Load reference trajectories
# ------------------------------------------------------------

references = {}

for f in glob.glob(os.path.join(FOLDER, "*_reference.json")):
    with open(f) as file:
        data = json.load(file)

    traj = np.array(data["reference_trajectory"], dtype=float)

    # Reference yaw is degrees -> radians
    traj[:, 3] = np.deg2rad(traj[:, 3])

    references[data["episode_key"]] = traj


# ------------------------------------------------------------
# Evaluate predictions
# ------------------------------------------------------------

sr = []
osr = []
ne = []
ndtw = []

prediction_files = glob.glob(
    os.path.join(FOLDER, "*.json.json")
)

for f in prediction_files:

    with open(f) as file:
        data = json.load(file)

    episode = data["episode_key"]

    if episode not in references:
        print(f"Missing reference: {episode}")
        continue

    pred = np.array(data["trajectory"], dtype=float)
    target = np.array(data["target_coords"], dtype=float)
    ref = references[episode]

    # --------------------------------------------------------
    # SR
    # --------------------------------------------------------

    final_dist = np.linalg.norm(pred[-1, :3] - target[:3])
    final_yaw = yaw_diff(pred[-1, 3], target[3])

    success = (
        final_dist < SUCCESS_DIST
        and final_yaw < SUCCESS_YAW
    )

    sr.append(success)

    # --------------------------------------------------------
    # OSR
    # --------------------------------------------------------

    oracle = any(
        np.linalg.norm(p[:3] - target[:3]) < SUCCESS_DIST
        and yaw_diff(p[3], target[3]) < SUCCESS_YAW
        for p in pred
    )

    osr.append(oracle)

    # --------------------------------------------------------
    # NE
    # --------------------------------------------------------

    ne.append(final_dist)

    # --------------------------------------------------------
    # NDTW
    # --------------------------------------------------------

    L = path_length(ref)
    theta = rotation_length(ref)

    spatial_dtw = dtw(
        ref[:, :3],
        pred[:, :3],
        lambda a, b: np.linalg.norm(a - b)
    )

    rotational_dtw = dtw(
        ref[:, 3],
        pred[:, 3],
        lambda a, b: yaw_diff(a, b)
    )

    spatial_ndtw = (
        math.exp(-spatial_dtw / (L * DTH))
        if L > 0 else 0
    )

    rotational_ndtw = (
        math.exp(-rotational_dtw / (theta * DTH))
        if theta > 0 else 0
    )

    Le = L / MEAN_PATH_LENGTH
    Theta_e = theta / MEAN_ROTATION

    if Le + Theta_e > 0:
        final_ndtw = (
            Le * spatial_ndtw +
            Theta_e * rotational_ndtw
        ) / (Le + Theta_e)
    else:
        final_ndtw = 1.0

    ndtw.append(final_ndtw)


# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------

print("\nIndoorUAV VLA Evaluation")
print("------------------------")
print(f"Trajectories: {len(sr)}")
print(f"SR:            {np.mean(sr):.4f} ({100*np.mean(sr):.2f}%)")
print(f"OSR:           {np.mean(osr):.4f} ({100*np.mean(osr):.2f}%)")
print(f"NE:            {np.mean(ne):.4f} m")
print(f"NDTW:          {np.mean(ndtw):.4f}")


IndoorUAV VLA Evaluation
------------------------
Trajectories: 100
SR:            0.1600 (16.00%)
OSR:           0.1600 (16.00%)
NE:            2.7604 m
NDTW:          0.1183
